In [41]:
# Existing imports...
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from statsmodels.tsa.stattools import adfuller
from matplotlib.backends.backend_pdf import PdfPages
import plotly.graph_objects as go

# Additional imports for Dash
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
from fpdf import FPDF
from datetime import datetime

# ---------------------------------------------
# 1. Load & Clean Data
# ---------------------------------------------
df = pd.read_csv("data_GPU.csv")
df.columns = [str(col).strip() for col in df.columns]

if df.columns[0].startswith("Unnamed"):
    df.rename(columns={df.columns[0]: "Metric"}, inplace=True)

id_vars = ['Metric', 'Select Stack By', 'Pivot By', 'And Pivot By']
value_vars = [col for col in df.columns if col not in id_vars]

df_melted = df.melt(id_vars=id_vars, value_vars=value_vars, var_name='MonthStr', value_name='Value')
df_melted['MonthStart'] = pd.to_datetime(df_melted['MonthStr'], errors='coerce')
df_melted['Value'] = pd.to_numeric(df_melted['Value'], errors='coerce')
df_melted.dropna(subset=['MonthStart', 'Value'], inplace=True)

metrics_of_interest = ['Instance GPU', 'Host Equivalents']
df_melted = df_melted[df_melted['Metric'].isin(metrics_of_interest)]
df_melted = df_melted[df_melted['MonthStart'] >= pd.to_datetime("2025-03-01")]

df_agg = df_melted.groupby(['Metric', 'Select Stack By', 'Pivot By', 'MonthStart'])['Value'] \
                  .quantile(0.95).reset_index(name='Percentile95')
df_agg['Group'] = df_agg['Select Stack By'] + " - " + df_agg['Pivot By']

df =  df_agg.copy()

# Create PDF for summary plots
os.makedirs("output", exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
pdf_path = f"output/EDA_Summary_Plots_{ts}.pdf"
summary_path = f"output/EDA_Summary_{ts}.csv"
summary_pdf_path = f"output/EDA_Text_Summary_{ts}.pdf"

summary_records = []

with PdfPages(pdf_path) as pdf:
    for (dc, cust_type), group in df.groupby(['Select Stack By', 'Pivot By']):
        for metric in group['Metric'].unique():
            subset = group[group['Metric'] == metric]
            ts_data = subset.set_index('MonthStart')['Percentile95'].sort_index()

            result = {}
            try:
                adf_stat, p_value, _, _, _, _ = adfuller(ts_data.dropna())
                result['p-value'] = p_value
                result['Insight'] = 'Stationary' if p_value < 0.05 else 'Non-Stationary'
            except Exception:
                result['p-value'] = None
                result['Insight'] = 'ADF Test Failed'

            summary_records.append({
                'Datacenter': dc,
                'Customer Type': cust_type,
                'Metric': metric,
                'ADF P-Value': result['p-value'],
                'Insight': result['Insight']
            })

            plt.figure(figsize=(10, 5))
            plt.plot(ts_data, marker='o')
            plt.title(f"{dc} | {cust_type} | {metric}")
            plt.xlabel("MonthStart")
            plt.ylabel("95th Percentile")
            plt.grid(True)

            if result['p-value'] is not None:
                text = f"ADF p-value: {result['p-value']:.4f}\nInsight: {result['Insight']}"
            else:
                text = f"ADF p-value: N/A\nInsight: {result['Insight']}"

            plt.figtext(0.15, 0.8, text)
            pdf.savefig()
            plt.close()

summary_df = pd.DataFrame(summary_records)
summary_df.to_csv(summary_path, index=False)

class PDF(FPDF):
    def header(self):
        self.set_font("Arial", "B", 12)
        self.cell(200, 10, "EDA Summary Report", ln=True, align="C")

    def add_table(self, data):
        self.set_font("Arial", size=10)
        self.ln(5)
        col_widths = [40, 40, 40, 40, 30]
        headers = ["Datacenter", "Customer Type", "Metric", "Insight", "ADF P-Value"]
        for i, header in enumerate(headers):
            self.cell(col_widths[i], 8, header, border=1)
        self.ln()
        for _, row in data.iterrows():
            self.cell(col_widths[0], 8, str(row["Datacenter"]), border=1)
            self.cell(col_widths[1], 8, str(row["Customer Type"]), border=1)
            self.cell(col_widths[2], 8, str(row["Metric"]), border=1)
            self.cell(col_widths[3], 8, str(row["Insight"]), border=1)
            self.cell(col_widths[4], 8, f"{row['ADF P-Value']:.4f}" if pd.notnull(row['ADF P-Value']) else "N/A", border=1)
            self.ln()

pdf = PDF()
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_table(summary_df)
pdf.output(summary_pdf_path)

# -----------------------
# Start Dash App
# -----------------------
app = Dash(__name__)
server = app.server

# Dropdown options
dc_unique = df['Select Stack By'].dropna().unique()
cust_unique = df['Pivot By'].dropna().unique()
metric_unique = df['Metric'].dropna().unique()

dc_options = [{'label': dc, 'value': dc} for dc in dc_unique] + [{'label': 'All DCs', 'value': 'All DCs'}]
cust_options = [{'label': c, 'value': c} for c in cust_unique]
metric_options = [{'label': m, 'value': m} for m in metric_unique]

app.layout = html.Div([
    html.H1("Time Series EDA Dashboard", style={'textAlign': 'center'}),

    html.Div([
        html.Div([
            html.Label("Select Customer Type:"),
            dcc.Dropdown(
                options=cust_options,
                value=[cust_unique[0]],
                multi=True,
                id='cust-type'
            )
        ], style={'width': '32%', 'display': 'inline-block'}),

        html.Div([
            html.Label("Select Datacentre:"),
            dcc.Dropdown(
                options=dc_options,
                value=[dc_unique[0]],
                multi=True,
                id='dc'
            )
        ], style={'width': '32%', 'display': 'inline-block'}),

        html.Div([
            html.Label("Select Metric:"),
            dcc.Dropdown(
                options=metric_options,
                value=metric_unique[0],
                id='metric-filter',
                clearable=False
            )
        ], style={'width': '32%', 'display': 'inline-block'}),
    ], style={'margin-bottom': '20px'}),

    dcc.Graph(id='ts-graph')
])

@app.callback(
    Output('ts-graph', 'figure'),
    Input('cust-type', 'value'),
    Input('dc', 'value'),
    Input('metric-filter', 'value')
)
def update_graph(selected_cust_types, selected_dcs, selected_metric):
    filtered_df = df[
        (df['Pivot By'].isin(selected_cust_types)) &
        (df['Metric'] == selected_metric)
    ]

    fig = go.Figure()

    # Handle 'All DCs'
    if 'All DCs' in selected_dcs:
        for cust in selected_cust_types:
            sub_df = filtered_df[filtered_df['Pivot By'] == cust]
            agg_df = sub_df.groupby('MonthStart')['Percentile95'].sum().reset_index()
            fig.add_trace(go.Scatter(
                x=agg_df['MonthStart'],
                y=agg_df['Percentile95'],
                mode='lines+markers',
                name=f"All DCs - {cust}"
            ))
        selected_dcs = [dc for dc in selected_dcs if dc != 'All DCs']

    # Individual DC + Customer Type combinations
    for dc in selected_dcs:
        for cust in selected_cust_types:
            sub_df = filtered_df[(filtered_df['Select Stack By'] == dc) & (filtered_df['Pivot By'] == cust)]
            if not sub_df.empty:
                fig.add_trace(go.Scatter(
                    x=sub_df['MonthStart'],
                    y=sub_df['Percentile95'],
                    mode='lines+markers',
                    name=f"{dc} - {cust}"
                ))

    fig.update_layout(
        title=f"{selected_metric} - 95th Percentile by Datacentre and Customer Type",
        xaxis_title="Month",
        yaxis_title="95th Percentile",
        legend_title="Group",
        hovermode="x unified"
    )

    return fig

if __name__ == '__main__':
    app.run(debug=True)

print(f"\n✅ All outputs generated successfully!")
print(f"📄 Plots PDF: {pdf_path}")
print(f"📊 Summary CSV: {summary_path}")
print(f"📄 Summary PDF: {summary_pdf_path}")



✅ All outputs generated successfully!
📄 Plots PDF: output/EDA_Summary_Plots_20250723_200315.pdf
📊 Summary CSV: output/EDA_Summary_20250723_200315.csv
📄 Summary PDF: output/EDA_Text_Summary_20250723_200315.pdf


In [42]:
df_agg.to_excel('dump.xlsx', index = False)

In [46]:
df_melted['MonthStr'].unique()

array(['03-01-2025', '03-02-2025', '03-03-2025', '03-04-2025',
       '03-05-2025', '03-06-2025', '03-07-2025', '03-08-2025',
       '03-09-2025', '03-10-2025', '03-11-2025', '03-12-2025',
       '04-01-2025', '04-02-2025', '04-03-2025', '04-04-2025',
       '04-05-2025', '04-06-2025', '04-07-2025', '04-08-2025',
       '04-09-2025', '04-10-2025', '04-11-2025', '04-12-2025',
       '05-01-2025', '05-02-2025', '05-03-2025', '05-04-2025',
       '05-05-2025', '05-06-2025', '05-07-2025', '05-08-2025',
       '05-09-2025', '05-10-2025', '05-11-2025', '05-12-2025',
       '06-01-2025', '06-02-2025', '06-03-2025', '06-04-2025',
       '06-05-2025', '06-06-2025', '06-07-2025', '06-08-2025',
       '06-09-2025', '06-10-2025', '06-11-2025', '06-12-2025',
       '07-01-2025', '07-02-2025', '07-03-2025', '07-04-2025',
       '07-05-2025', '07-06-2025', '07-07-2025', '07-08-2025',
       '07-09-2025', '07-10-2025', '07-11-2025', '07-12-2025'],
      dtype=object)

In [53]:
# --- Imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from statsmodels.tsa.stattools import adfuller
from matplotlib.backends.backend_pdf import PdfPages
import plotly.graph_objects as go

# Additional for Dash
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
from fpdf import FPDF
from datetime import datetime

# ---------------------------------------------
# 1. Load & Clean Data
# ---------------------------------------------

# Function to clean and parse mixed-format date columns
def parse_mixed_date_columns(df, id_vars):
    all_cols = list(df.columns)
    date_cols_raw = [col for col in all_cols if col not in id_vars]
    
    fixed_date_cols = {}
    unparsed_cols = []
    for col in date_cols_raw:
        try:
            parsed = pd.to_datetime(col, errors='raise')
            fixed_date_cols[col] = parsed
        except:
            unparsed_cols.append(col)

    # Rename successfully parsed columns
    df_renamed = df.rename(columns=fixed_date_cols)

    # Drop unparseable ones (can log or warn)
    if unparsed_cols:
        print(f"⚠️ Dropping unparseable columns: {unparsed_cols}")
        df_renamed.drop(columns=unparsed_cols, inplace=True)

    return df_renamed, list(fixed_date_cols.values())

# Load raw dataset
raw_df = pd.read_csv("data_GPU.csv")
raw_df.columns = [str(col).strip() for col in raw_df.columns]

# Rename first column if Unnamed
if raw_df.columns[0].startswith("Unnamed"):
    raw_df.rename(columns={raw_df.columns[0]: "Metric"}, inplace=True)

# Identify ID and date columns
id_vars = ['Metric', 'Select Stack By', 'Pivot By', 'And Pivot By']
raw_df, parsed_date_cols = parse_mixed_date_columns(raw_df, id_vars)

# Melt to long format
df_melted = raw_df.melt(id_vars=id_vars, value_vars=parsed_date_cols,
                        var_name='MonthStart', value_name='Value')

df_melted['MonthStart'] = pd.to_datetime(df_melted['MonthStart'], errors='coerce')
df_melted['Value'] = pd.to_numeric(df_melted['Value'], errors='coerce')
df_melted.dropna(subset=['MonthStart', 'Value'], inplace=True)

# Filter metrics and date range
metrics_of_interest = ['Instance GPU', 'Host Equivalents']
df_melted = df_melted[df_melted['Metric'].isin(metrics_of_interest)]
df_melted = df_melted[df_melted['MonthStart'] >= pd.to_datetime("2025-03-01")]

# Aggregation
df_agg = df_melted.groupby(['Metric', 'Select Stack By', 'Pivot By', 'MonthStart'])['Value'] \
                  .quantile(0.95).reset_index(name='Percentile95')
df_agg['Group'] = df_agg['Select Stack By'] + " - " + df_agg['Pivot By']
df = df_agg.copy()

# ---------------------------------------------
# 2. Generate Summary PDF + CSV
# ---------------------------------------------
os.makedirs("output", exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
pdf_path = f"output/EDA_Summary_Plots_{ts}.pdf"
summary_path = f"output/EDA_Summary_{ts}.csv"
summary_pdf_path = f"output/EDA_Text_Summary_{ts}.pdf"

summary_records = []

with PdfPages(pdf_path) as pdf:
    for (dc, cust_type), group in df.groupby(['Select Stack By', 'Pivot By']):
        for metric in group['Metric'].unique():
            subset = group[group['Metric'] == metric]
            ts_data = subset.set_index('MonthStart')['Percentile95'].sort_index()

            result = {}
            try:
                adf_stat, p_value, *_ = adfuller(ts_data.dropna())
                result['p-value'] = p_value
                result['Insight'] = 'Stationary' if p_value < 0.05 else 'Non-Stationary'
            except:
                result['p-value'] = None
                result['Insight'] = 'ADF Test Failed'

            summary_records.append({
                'Datacenter': dc,
                'Customer Type': cust_type,
                'Metric': metric,
                'ADF P-Value': result['p-value'],
                'Insight': result['Insight']
            })

            # Plotting
            plt.figure(figsize=(10, 5))
            plt.plot(ts_data, marker='o')
            plt.title(f"{dc} | {cust_type} | {metric}")
            plt.xlabel("MonthStart")
            plt.ylabel("95th Percentile")
            plt.grid(True)

            annotation = f"ADF p-value: {result['p-value']:.4f}" if result['p-value'] else "ADF p-value: N/A"
            plt.figtext(0.15, 0.8, f"{annotation}\nInsight: {result['Insight']}")
            pdf.savefig()
            plt.close()

summary_df = pd.DataFrame(summary_records)
summary_df.to_csv(summary_path, index=False)

class PDF(FPDF):
    def header(self):
        self.set_font("Arial", "B", 12)
        self.cell(200, 10, "EDA Summary Report", ln=True, align="C")

    def add_table(self, data):
        self.set_font("Arial", size=10)
        self.ln(5)
        col_widths = [40, 40, 40, 40, 30]
        headers = ["Datacenter", "Customer Type", "Metric", "Insight", "ADF P-Value"]
        for i, header in enumerate(headers):
            self.cell(col_widths[i], 8, header, border=1)
        self.ln()
        for _, row in data.iterrows():
            self.cell(col_widths[0], 8, str(row["Datacenter"]), border=1)
            self.cell(col_widths[1], 8, str(row["Customer Type"]), border=1)
            self.cell(col_widths[2], 8, str(row["Metric"]), border=1)
            self.cell(col_widths[3], 8, str(row["Insight"]), border=1)
            self.cell(col_widths[4], 8, f"{row['ADF P-Value']:.4f}" if pd.notnull(row['ADF P-Value']) else "N/A", border=1)
            self.ln()

pdf = PDF()
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_table(summary_df)
pdf.output(summary_pdf_path)

# ---------------------------------------------
# 3. Dash App for Time Series EDA
# ---------------------------------------------
app = Dash(__name__)
server = app.server

# Dropdown setup
dc_unique = df['Select Stack By'].dropna().unique()
cust_unique = df['Pivot By'].dropna().unique()
metric_unique = df['Metric'].dropna().unique()

dc_options = [{'label': dc, 'value': dc} for dc in dc_unique] + [{'label': 'All DCs', 'value': 'All DCs'}]
cust_options = [{'label': c, 'value': c} for c in cust_unique]
metric_options = [{'label': m, 'value': m} for m in metric_unique]

app.layout = html.Div([
    html.H1("Time Series EDA Dashboard", style={'textAlign': 'center'}),

    html.Div([
        html.Div([
            html.Label("Select Customer Type:"),
            dcc.Dropdown(
                options=cust_options,
                value=[cust_unique[0]],
                multi=True,
                id='cust-type'
            )
        ], style={'width': '32%', 'display': 'inline-block'}),

        html.Div([
            html.Label("Select Datacentre:"),
            dcc.Dropdown(
                options=dc_options,
                value=[dc_unique[0]],
                multi=True,
                id='dc'
            )
        ], style={'width': '32%', 'display': 'inline-block'}),

        html.Div([
            html.Label("Select Metric:"),
            dcc.Dropdown(
                options=metric_options,
                value=metric_unique[0],
                id='metric-filter',
                clearable=False
            )
        ], style={'width': '32%', 'display': 'inline-block'}),
    ], style={'margin-bottom': '20px'}),

    dcc.Graph(id='ts-graph')
])

@app.callback(
    Output('ts-graph', 'figure'),
    Input('cust-type', 'value'),
    Input('dc', 'value'),
    Input('metric-filter', 'value')
)
def update_graph(selected_cust_types, selected_dcs, selected_metric):
    filtered_df = df[
        (df['Pivot By'].isin(selected_cust_types)) &
        (df['Metric'] == selected_metric)
    ]

    fig = go.Figure()

    if 'All DCs' in selected_dcs:
        for cust in selected_cust_types:
            sub_df = filtered_df[filtered_df['Pivot By'] == cust]
            agg_df = sub_df.groupby('MonthStart')['Percentile95'].sum().reset_index()
            fig.add_trace(go.Scatter(
                x=agg_df['MonthStart'],
                y=agg_df['Percentile95'],
                mode='lines+markers',
                name=f"All DCs - {cust}"
            ))
        selected_dcs = [dc for dc in selected_dcs if dc != 'All DCs']

    for dc in selected_dcs:
        for cust in selected_cust_types:
            sub_df = filtered_df[
                (filtered_df['Select Stack By'] == dc) & 
                (filtered_df['Pivot By'] == cust)
            ]
            if not sub_df.empty:
                fig.add_trace(go.Scatter(
                    x=sub_df['MonthStart'],
                    y=sub_df['Percentile95'],
                    mode='lines+markers',
                    name=f"{dc} - {cust}"
                ))

    fig.update_layout(
        title=f"{selected_metric} - 95th Percentile by Datacentre and Customer Type",
        xaxis_title="Month",
        yaxis_title="95th Percentile",
        legend_title="Group",
        hovermode="x unified"
    )

    return fig

if __name__ == '__main__':
    app.run(debug=True)

print(f"\n✅ All outputs generated successfully!")
print(f"📄 Plots PDF: {pdf_path}")
print(f"📊 Summary CSV: {summary_path}")
print(f"📄 Summary PDF: {summary_pdf_path}")



✅ All outputs generated successfully!
📄 Plots PDF: output/EDA_Summary_Plots_20250724_112546.pdf
📊 Summary CSV: output/EDA_Summary_20250724_112546.csv
📄 Summary PDF: output/EDA_Text_Summary_20250724_112546.pdf


In [50]:
raw_df.head(20)

,Metric,Select Stack By,Pivot By,And Pivot By,2024-08-01 00:00:00,2024-08-02 00:00:00,2024-08-03 00:00:00,2024-08-04 00:00:00,2024-08-05 00:00:00,2024-08-06 00:00:00,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
0,Host Equivalents,"ATLANTA (Atlanta, GA, USA)",External,All,1.0,1.0,1.0,1.0,1.0,1.0,...,1,1,1,1,1,1,1,1,1,1
1,Host Equivalents,"ATLANTA (Atlanta, GA, USA)",Internal,All,1.0,1.0,1.0,1.0,1.0,1.0,...,0,0,0,0,0,0,0,0,0,0
2,Host Equivalents,"BOM1 (Mumbai, IN)",External,All,NaN,NaN,NaN,NaN,NaN,NaN,...,5,8,8,8,8,8,8,6,8,5
3,Host Equivalents,"BOM1 (Mumbai, IN)",Internal,All,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,1,1,1
4,Host Equivalents,"FRA4 (Frankfurt, DE)",External,All,NaN,NaN,NaN,NaN,NaN,NaN,...,14,14,14,13,12,14,14,14,13,13
5,Host Equivalents,"FRA4 (Frankfurt, DE)",Internal,All,NaN,NaN,NaN,NaN,NaN,NaN,...,3,3,3,3,3,3,3,3,3,3
6,Host Equivalents,"FRANKFURT (Frankfurt, DE)",External,All,1.0,1.0,1.0,1.0,1.0,1.0,...,0,0,0,0,0,0,0,0,0,0
7,Host Equivalents,"FRANKFURT (Frankfurt, DE)",Internal,All,1.0,1.0,1.0,1.0,1.0,1.0,...,1,1,1,1,1,1,1,1,1,1
8,Host Equivalents,"MUM1 (Mumbai, India)",External,All,1.0,1.0,1.0,1.0,1.0,1.0,...,1,1,1,1,1,1,1,1,1,1
9,Host Equivalents,"MUM1 (Mumbai, India)",Internal,All,1.0,1.0,1.0,1.0,1.0,1.0,...,0,0,0,0,0,0,0,0,0,0


In [57]:
# --- Imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from statsmodels.tsa.stattools import adfuller
from matplotlib.backends.backend_pdf import PdfPages
import plotly.graph_objects as go

# Additional for Dash
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
from fpdf import FPDF
from datetime import datetime
from threading import Thread
import webbrowser

# ---------------------------------------------
# 1. Load & Clean Data
# ---------------------------------------------

# Function to clean and parse mixed-format date columns
def parse_mixed_date_columns(df, id_vars):
    all_cols = list(df.columns)
    date_cols_raw = [col for col in all_cols if col not in id_vars]

    fixed_date_cols = {}
    unparsed_cols = []
    for col in date_cols_raw:
        try:
            parsed = pd.to_datetime(col, errors='raise')
            fixed_date_cols[col] = parsed
        except:
            unparsed_cols.append(col)

    # Rename successfully parsed columns
    df_renamed = df.rename(columns=fixed_date_cols)

    # Drop unparseable ones (can log or warn)
    if unparsed_cols:
        print(f"⚠️ Dropping unparseable columns: {unparsed_cols}")
        df_renamed.drop(columns=unparsed_cols, inplace=True)

    return df_renamed, list(fixed_date_cols.values())

# Load raw dataset
raw_df = pd.read_csv("data_GPU.csv")
raw_df.columns = [str(col).strip() for col in raw_df.columns]

# Rename first column if Unnamed
if raw_df.columns[0].startswith("Unnamed"):
    raw_df.rename(columns={raw_df.columns[0]: "Metric"}, inplace=True)

# Identify ID and date columns
id_vars = ['Metric', 'Select Stack By', 'Pivot By', 'And Pivot By']
raw_df, parsed_date_cols = parse_mixed_date_columns(raw_df, id_vars)

# Melt to long format
df_melted = raw_df.melt(id_vars=id_vars, value_vars=parsed_date_cols,
                        var_name='MonthStart', value_name='Value')

df_melted['MonthStart'] = pd.to_datetime(df_melted['MonthStart'], errors='coerce')
df_melted['Value'] = pd.to_numeric(df_melted['Value'], errors='coerce')
df_melted.dropna(subset=['MonthStart', 'Value'], inplace=True)

# Filter metrics and date range
metrics_of_interest = ['Instance GPU', 'Host Equivalents']
df_melted = df_melted[df_melted['Metric'].isin(metrics_of_interest)]
df_melted = df_melted[df_melted['MonthStart'] >= pd.to_datetime("2025-03-01")]

# Aggregation
df_agg = df_melted.groupby(['Metric', 'Select Stack By', 'Pivot By', 'MonthStart'])['Value'] \
                  .quantile(0.95).reset_index(name='Percentile95')
df_agg['Group'] = df_agg['Select Stack By'] + " - " + df_agg['Pivot By']
df = df_agg.copy()

# ---------------------------------------------
# 2. Generate Summary PDF + CSV
# ---------------------------------------------
os.makedirs("output", exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
pdf_path = f"output/EDA_Summary_Plots_{ts}.pdf"
summary_path = f"output/EDA_Summary_{ts}.csv"
summary_pdf_path = f"output/EDA_Text_Summary_{ts}.pdf"

summary_records = []

with PdfPages(pdf_path) as pdf:
    for (dc, cust_type), group in df.groupby(['Select Stack By', 'Pivot By']):
        for metric in group['Metric'].unique():
            subset = group[group['Metric'] == metric]
            ts_data = subset.set_index('MonthStart')['Percentile95'].sort_index()

            result = {}
            try:
                adf_stat, p_value, *_ = adfuller(ts_data.dropna())
                result['p-value'] = p_value
                result['Insight'] = 'Stationary' if p_value < 0.05 else 'Non-Stationary'
            except:
                result['p-value'] = None
                result['Insight'] = 'ADF Test Failed'

            summary_records.append({
                'Datacenter': dc,
                'Customer Type': cust_type,
                'Metric': metric,
                'ADF P-Value': result['p-value'],
                'Insight': result['Insight']
            })

            # Plotting
            plt.figure(figsize=(10, 5))
            plt.plot(ts_data, marker='o')
            plt.title(f"{dc} | {cust_type} | {metric}")
            plt.xlabel("MonthStart")
            plt.ylabel("95th Percentile")
            plt.grid(True)

            annotation = f"ADF p-value: {result['p-value']:.4f}" if result['p-value'] else "ADF p-value: N/A"
            plt.figtext(0.15, 0.8, f"{annotation}\nInsight: {result['Insight']}")
            pdf.savefig()
            plt.close()

summary_df = pd.DataFrame(summary_records)
summary_df.to_csv(summary_path, index=False)

class PDF(FPDF):
    def header(self):
        self.set_font("Arial", "B", 12)
        self.cell(200, 10, "EDA Summary Report", ln=True, align="C")

    def add_table(self, data):
        self.set_font("Arial", size=10)
        self.ln(5)
        col_widths = [40, 40, 40, 40, 30]
        headers = ["Datacenter", "Customer Type", "Metric", "Insight", "ADF P-Value"]
        for i, header in enumerate(headers):
            self.cell(col_widths[i], 8, header, border=1)
        self.ln()
        for _, row in data.iterrows():
            self.cell(col_widths[0], 8, str(row["Datacenter"]), border=1)
            self.cell(col_widths[1], 8, str(row["Customer Type"]), border=1)
            self.cell(col_widths[2], 8, str(row["Metric"]), border=1)
            self.cell(col_widths[3], 8, str(row["Insight"]), border=1)
            self.cell(col_widths[4], 8, f"{row['ADF P-Value']:.4f}" if pd.notnull(row['ADF P-Value']) else "N/A", border=1)
            self.ln()

pdf = PDF()
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_table(summary_df)
pdf.output(summary_pdf_path)

# ---------------------------------------------
# 3. Dash Apps
# ---------------------------------------------
app1 = Dash(__name__, title="Time Series EDA Dashboard")
server = app1.server

# Dropdown setup
dc_unique = df['Select Stack By'].dropna().unique()
cust_unique = df['Pivot By'].dropna().unique()
metric_unique = df['Metric'].dropna().unique()

dc_options = [{'label': dc, 'value': dc} for dc in dc_unique] + [{'label': 'All DCs', 'value': 'All DCs'}]
cust_options = [{'label': c, 'value': c} for c in cust_unique]
metric_options = [{'label': m, 'value': m} for m in metric_unique]

app1.layout = html.Div([
    html.H1("Time Series EDA Dashboard", style={'textAlign': 'center'}),

    html.Div([
        html.Div([
            html.Label("Select Customer Type:"),
            dcc.Dropdown(
                options=cust_options,
                value=[cust_unique[0]],
                multi=True,
                id='cust-type'
            )
        ], style={'width': '32%', 'display': 'inline-block'}),

        html.Div([
            html.Label("Select Datacentre:"),
            dcc.Dropdown(
                options=dc_options,
                value=[dc_unique[0]],
                multi=True,
                id='dc'
            )
        ], style={'width': '32%', 'display': 'inline-block'}),

        html.Div([
            html.Label("Select Metric:"),
            dcc.Dropdown(
                options=metric_options,
                value=metric_unique[0],
                id='metric-filter',
                clearable=False
            )
        ], style={'width': '32%', 'display': 'inline-block'}),
    ], style={'margin-bottom': '20px'}),

    dcc.Graph(id='ts-graph')
])

@app1.callback(
    Output('ts-graph', 'figure'),
    Input('cust-type', 'value'),
    Input('dc', 'value'),
    Input('metric-filter', 'value')
)
def update_graph(selected_cust_types, selected_dcs, selected_metric):
    filtered_df = df[
        (df['Pivot By'].isin(selected_cust_types)) &
        (df['Metric'] == selected_metric)
    ]

    fig = go.Figure()

    if 'All DCs' in selected_dcs:
        for cust in selected_cust_types:
            sub_df = filtered_df[filtered_df['Pivot By'] == cust]
            agg_df = sub_df.groupby('MonthStart')['Percentile95'].sum().reset_index()
            fig.add_trace(go.Scatter(
                x=agg_df['MonthStart'],
                y=agg_df['Percentile95'],
                mode='lines+markers',
                name=f"All DCs - {cust}"
            ))
        selected_dcs = [dc for dc in selected_dcs if dc != 'All DCs']

    for dc in selected_dcs:
        for cust in selected_cust_types:
            sub_df = filtered_df[
                (filtered_df['Select Stack By'] == dc) & 
                (filtered_df['Pivot By'] == cust)
            ]
            if not sub_df.empty:
                fig.add_trace(go.Scatter(
                    x=sub_df['MonthStart'],
                    y=sub_df['Percentile95'],
                    mode='lines+markers',
                    name=f"{dc} - {cust}"
                ))

    fig.update_layout(
        title=f"{selected_metric} - 95th Percentile by Datacentre and Customer Type",
        xaxis_title="Month",
        yaxis_title="95th Percentile",
        legend_title="Group",
        hovermode="x unified"
    )

    return fig

app2 = Dash(__name__, title="Internal vs External Trends")
app2.layout = html.Div([
    html.H2("Internal vs External Trend (Across DCs)", style={"textAlign": "center"}),

    html.Div([
        html.Label("Select Metric:"),
        dcc.Dropdown(
            id='metric-type',
            options=metric_options,
            value=metric_unique[0]
        ),
        html.Label("Select Data Centers:"),
        dcc.Dropdown(
            id='multi-dc',
            options=[{'label': d, 'value': d} for d in dc_unique],
            value=[dc_unique[0]],
            multi=True
        )
    ], style={'width': '90%', 'margin': '0 auto'}),

    dcc.Graph(id='stacked-plot')
])

@app2.callback(
    Output('stacked-plot', 'figure'),
    Input('metric-type', 'value'),
    Input('multi-dc', 'value')
)
def update_stacked(metric, dcs):
    dff = df[(df['Metric'] == metric) & (df['Select Stack By'].isin(dcs))]
    dff_grouped = dff.groupby(['MonthStart', 'Pivot By'])['Percentile95'].sum().reset_index()
    fig = px.line(dff_grouped, x='MonthStart', y='Percentile95', color='Pivot By', title=f"{metric} Usage by Customer Type")
    fig.update_traces(mode="lines+markers")
    return fig

if __name__ == '__main__':
    Thread(target=lambda: app1.run_server(port=8050)).start()
    Thread(target=lambda: app2.run_server(port=8051)).start()
    webbrowser.open("http://localhost:8050")
    webbrowser.open("http://localhost:8051")

print(f"\n✅ All outputs generated successfully!")
print(f"📄 Plots PDF: {pdf_path}")
print(f"📈 Summary CSV: {summary_path}")
print(f"📄 Summary PDF: {summary_pdf_path}")

Exception in thread Thread-10195 (<lambda>):
Traceback (most recent call last):
  File "C:\Users\swappati\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
Exception in thread Thread-10196 (<lambda>):
Traceback (most recent call last):
  File "C:\Users\swappati\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "C:\Users\swappati\PycharmProjects\GPU_Forecasting\.venv\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    self.run()
  File "C:\Users\swappati\PycharmProjects\GPU_Forecasting\.venv\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\swappati\AppData\Local\Programs\Python\Python310\lib\threading.py", line 953, in run
    _threading_Thread_run(self)
  File "C:\Users\swappati\AppData\Local\Programs\Python\Python310\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  


✅ All outputs generated successfully!
📄 Plots PDF: output/EDA_Summary_Plots_20250724_171929.pdf
📈 Summary CSV: output/EDA_Summary_20250724_171929.csv
📄 Summary PDF: output/EDA_Text_Summary_20250724_171929.pdf
